# Produce Spleen Prediction CSVs

In [1]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    sprint_dir = candidate / "codes" / "sprint"
    if (sprint_dir / "prediction_export.py").exists():
        PROJECT_ROOT = candidate
        if str(PROJECT_ROOT) not in sys.path:
            sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise RuntimeError(f"Cannot find codes/sprint/prediction_export.py from {start}")

LEGACY_DIR = PROJECT_ROOT / "codes" / "_legacy_models" / "spleen"
if str(LEGACY_DIR) not in sys.path:
    sys.path.insert(0, str(LEGACY_DIR))

from codes.sprint.prediction_export import select_least_used_cuda_before_torch_import

select_least_used_cuda_before_torch_import()

Detected CUDA devices before torch import:
  physical=0 used=3358 MB / 11264 MB (29.8%) <-- selected as cuda:0


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from torch.utils.data import DataLoader

from codes.sprint.prediction_export import export_model_specs, find_project_root

# Legacy model & dataset classes
from codes._legacy_models.spleen.model import Model_C2, Model_RNA_Only
from codes._legacy_models.spleen.utils import SpleenMultimodalDataset, load_model_weights, set_seed

PROJECT_ROOT = find_project_root(Path.cwd())
print(f"Project root: {PROJECT_ROOT}")

set_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Project root: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github
Using device: cuda:0


In [4]:
# ---- Paths ----
TRAIN_H5AD = PROJECT_ROOT / "datas" / "spleen" / "mouse_spleen_1" / "mouse_spleen_1_Processed.h5ad"
VAL_H5AD = PROJECT_ROOT / "datas" / "spleen" / "mouse_spleen_2" / "mouse_spleen_2_Processed.h5ad"
MODEL_SAVE_ROOT = PROJECT_ROOT /"datas"/ "models" / "spleen"
OUTPUT_DIR = PROJECT_ROOT /"datas"/ "outputs" / "spleen"

MODEL_SPECS = [
    {"label": "C2", "model_class": Model_C2,
     "model_dir_candidates": ["C2_20260624", "C2"],
     "output_prefix": "spleen_C2"},
]
BATCH_SIZE = 8

for required_path in [TRAIN_H5AD, VAL_H5AD, MODEL_SAVE_ROOT]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
ad_train = sc.read_h5ad(TRAIN_H5AD, backed="r")
ad_val = sc.read_h5ad(VAL_H5AD, backed="r")
p_train = [str(x) for x in list(ad_train.uns.get("protein_names", []))]
p_val = [str(x) for x in list(ad_val.uns.get("protein_names", []))]
common_proteins = sorted(set(p_train).intersection(p_val)) or None
del ad_train, ad_val

val_dataset = SpleenMultimodalDataset(str(VAL_H5AD), target_proteins=common_proteins)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

NUM_GENES = val_dataset.rna_data.shape[1]
NUM_TARGETS = val_dataset.protein_data.shape[1]
target_names = [str(x) for x in list(val_dataset.protein_names)]

print(f"Inference config: {NUM_GENES} genes, {NUM_TARGETS} proteins, "
      f"{len(val_dataset)} validation spots")
print(target_names)

📖 Loading data from: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/spleen/mouse_spleen_2/mouse_spleen_2_Processed.h5ad ...
Inference config: 32285 genes, 21 proteins, 2768 validation spots
['B220_CD45R', 'CD105', 'CD11b', 'CD163', 'CD169_siglec', 'CD19', 'CD20', 'CD29', 'CD3', 'CD31', 'CD38', 'CD4', 'CD68', 'CD8', 'EpCAM', 'F4_80', 'IgD', 'IgM', 'Ly6C', 'Ly6G', 'MadCAM1']


In [6]:
saved_df, pred_df, target_df = export_model_specs(
    MODEL_SPECS,
    MODEL_SAVE_ROOT,
    OUTPUT_DIR,
    val_loader,
    device,
    NUM_TARGETS,
    NUM_GENES,
    target_names,
    load_model_weights,
)

display(saved_df)
display(pred_df.head())
display(target_df.head())

[C2] saved predictions: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/spleen/spleen_C2_predictions.csv
[C2] saved targets:     /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/spleen/spleen_C2_targets.csv


,Model,Weights,ModelDir,Predictions,Targets,Rows,TargetsCount
0,C2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,C2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,2768,21


,test_index,B220_CD45R,CD105,CD11b,CD163,CD169_siglec,CD19,CD20,CD29,CD3,...,CD4,CD68,CD8,EpCAM,F4_80,IgD,IgM,Ly6C,Ly6G,MadCAM1
0,0,4.484493,4.037529,3.862385,3.815336,4.177335,4.471859,2.716582,4.334231,4.221838,...,4.123352,5.288543,4.175382,2.477851,4.907680,5.774501,3.149997,2.862500,3.071766,2.593440
1,1,4.585457,3.352146,3.166762,2.598948,4.013347,4.668681,2.448412,3.069897,4.201182,...,4.076071,3.916058,3.909107,2.156196,3.229875,6.121450,2.762103,2.455524,2.783569,2.466095
2,2,3.644956,3.690506,3.355476,3.895250,3.101252,3.601814,2.451026,3.770303,3.683856,...,3.400175,4.824631,3.894798,2.231695,4.918191,5.070199,2.649891,2.397872,2.795851,2.348441
3,3,4.260535,4.069267,3.834201,4.066531,3.769844,4.266745,2.663403,4.334231,4.236118,...,4.059740,5.286635,4.342666,2.523053,5.286021,5.653281,2.984030,2.722154,3.243817,2.728104
4,4,4.148006,3.954922,3.757103,3.954870,3.539342,4.095209,2.735927,4.324579,4.202549,...,3.948774,5.312016,4.337077,2.469940,5.216951,5.495825,2.977556,2.801685,3.120464,2.694715


,test_index,B220_CD45R,CD105,CD11b,CD163,CD169_siglec,CD19,CD20,CD29,CD3,...,CD4,CD68,CD8,EpCAM,F4_80,IgD,IgM,Ly6C,Ly6G,MadCAM1
0,0,4.248495,3.970292,3.367296,2.890372,4.158883,4.219508,2.639057,4.127134,4.143135,...,3.737670,5.023880,4.382027,1.791759,4.477337,5.765191,2.564949,2.397895,2.772589,2.833213
1,1,4.564348,3.178054,3.044523,2.564949,3.526361,4.624973,2.484907,3.091043,3.828641,...,3.850147,3.688879,3.737670,1.609438,2.944439,6.042633,2.890372,2.397895,2.397895,2.197225
2,2,3.433987,3.555348,3.218876,3.367296,3.258096,2.995732,2.397895,3.850147,3.496508,...,3.367296,4.897840,4.158883,2.302585,4.912655,4.753590,2.772589,1.945910,2.833213,1.945910
3,3,4.356709,4.762174,4.290460,5.030438,3.713572,4.262680,3.258096,4.912655,4.442651,...,4.499810,6.165418,4.795791,2.772589,6.186209,5.762052,4.304065,3.178054,3.663562,2.890372
4,4,4.369448,4.343805,3.871201,4.477337,3.737670,4.317488,3.258096,4.574711,4.234107,...,3.912023,6.030685,4.615120,2.890372,5.743003,5.579730,3.367296,2.995732,3.044523,3.258096
